# Wine Quality: clasificación con Naive Bayes, SMOTE y XGBoost

Este notebook reproduce el análisis realizado sobre el conjunto de datos **Wine Quality (red wine)**. Se estudia primero el problema multicategoría y posteriormente se transforma en un problema binario. Finalmente se compara XGBoost con y sin SMOTE y se realiza validación cruzada estratificada de 5 folds.

In [569]:
import numpy as np
from NBClass import NaiveBayes
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.naive_bayes import GaussianNB
from imblearn.over_sampling import RandomOverSampler
from imblearn.over_sampling import SMOTE
from xgboost import XGBRegressor
from sklearn.model_selection import StratifiedKFold, cross_validate

## 1. Carga y exploración inicial de los datos

En esta sección se carga el conjunto de datos y se revisa su estructura y la distribución de la variable objetivo `quality`.

In [570]:
df=pd.read_csv("data/winequality-red.csv", sep=";")

In [571]:
df.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


In [572]:
df['quality'].value_counts()

quality
5    681
6    638
7    199
4     53
8     18
3     10
Name: count, dtype: int64

In [573]:
X=df.drop('quality', axis=1)
X=X.values
y=df["quality"]
y=y.values

## 2. División inicial de los datos

Se separan las variables predictoras y la variable objetivo y se realiza una partición entrenamiento/prueba manteniendo la proporción de las clases mediante `stratify`.

In [574]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)


## 3. Naive Bayes: problema multicategoría

Se utiliza la implementación propia de Naive Bayes para predecir las categorías originales de calidad del vino.

In [575]:
modelo=NaiveBayes()

In [576]:
modelo.fit(X_train, y_train)

In [577]:
y_pred=modelo.predict(X_test)


In [578]:
pd.value_counts(y_test)

C:\Users\Pablo Duran\AppData\Local\Temp\ipykernel_32664\1894360198.py:1: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  pd.value_counts(y_test)


5    170
6    160
7     50
4     13
8      5
3      2
Name: count, dtype: int64

In [579]:
pd.value_counts(y_train)

C:\Users\Pablo Duran\AppData\Local\Temp\ipykernel_32664\4046760108.py:1: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  pd.value_counts(y_train)


5    511
6    478
7    149
4     40
8     13
3      8
Name: count, dtype: int64

In [580]:
accuracy = accuracy_score(y_test, y_pred)
matrix = confusion_matrix(y_test, y_pred)
print(accuracy)
print(classification_report(y_test, y_pred))
print(matrix) 

0.56
              precision    recall  f1-score   support

           3       0.00      0.00      0.00         2
           4       0.12      0.15      0.14        13
           5       0.68      0.66      0.67       170
           6       0.54      0.51      0.52       160
           7       0.49      0.54      0.51        50
           8       0.11      0.20      0.14         5

    accuracy                           0.56       400
   macro avg       0.33      0.34      0.33       400
weighted avg       0.57      0.56      0.57       400

[[  0   1   1   0   0   0]
 [  2   2   5   4   0   0]
 [  2   6 113  46   2   1]
 [  1   7  46  81  22   3]
 [  0   0   0  19  27   4]
 [  0   0   0   0   4   1]]


## 4. Naive Bayes con estandarización

Se estandarizan las variables predictoras y se vuelve a entrenar Naive Bayes para comparar el resultado.

In [581]:
estandardizador = StandardScaler()
X_train = estandardizador.fit_transform(X_train)
X_test = estandardizador.transform(X_test)

In [582]:
modelo2=NaiveBayes()
modelo2.fit(X_train, y_train)

In [583]:
y_pred2=modelo2.predict(X_test)

In [584]:
accuracy2 = accuracy_score(y_test, y_pred2)
matrix2 = confusion_matrix(y_test, y_pred2)
print(accuracy2)
print(classification_report(y_test, y_pred2))
print(matrix2)

0.56
              precision    recall  f1-score   support

           3       0.00      0.00      0.00         2
           4       0.12      0.15      0.14        13
           5       0.67      0.66      0.67       170
           6       0.54      0.51      0.52       160
           7       0.53      0.56      0.54        50
           8       0.10      0.20      0.13         5

    accuracy                           0.56       400
   macro avg       0.33      0.35      0.33       400
weighted avg       0.57      0.56      0.57       400

[[  0   1   1   0   0   0]
 [  2   2   5   4   0   0]
 [  2   6 112  47   2   1]
 [  1   7  48  81  20   3]
 [  0   0   0  17  28   5]
 [  0   0   0   1   3   1]]


## 5. Naive Bayes con Random Oversampling

Se aplica Random Oversampling únicamente al conjunto de entrenamiento para atender el desbalance de clases.

In [585]:
ros = RandomOverSampler(random_state=42)

X_train_ros, y_train_ros = ros.fit_resample(X_train,y_train)

In [586]:
modelo3=GaussianNB()
modelo3.fit(X_train_ros, y_train_ros)

,"priors priors: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
,"var_smoothing var_smoothing: float, default=1e-9Portion of the largest variance of all features that is added tovariances for calculation stability... versionadded:: 0.20",1e-09
Name,Type,Value
"class_count_ class_count_: ndarray of shape (n_classes,)number of training samples observed in each class.","ndarray[float64](6,)","[511.,511.,511.,511.,511.,511.]"
"class_prior_ class_prior_: ndarray of shape (n_classes,)probability of each class.","ndarray[float64](6,)","[0.17,0.17,0.17,0.17,0.17,0.17]"
"classes_ classes_: ndarray of shape (n_classes,)class labels known to the classifier.","ndarray[int64](6,)","[3,4,5,6,7,8]"
epsilon_ epsilon_: floatabsolute additive value to variances.,float64,2.395e-09
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,11
"theta_ theta_: ndarray of shape (n_classes, n_features)mean of each feature per class.","ndarray[float64](6, 11)","[[-0.07, 2.49,-0.65,..., 0.7 ,-0.7 ,-0.4 ], [-0.31, 0.9 ,-0.53,..., 0.53,-0.26,-0.08], [-0.1 , 0.27,-0.13,...,-0.02,-0.2 ,-0.5 ], [ 0.04,-0.16, 0. ,...,-0.01, 0.09, 0.17], [ 0.27,-0.7 , 0.47,...,-0.09, 0.47, 0.9 ], [-0.16,-0.54, 0.45,...,-0.03, 0.64, 1.69]]"
"var_ var_: ndarray of shape (n_classes, n_features)Variance of each feature per class... versionadded:: 1.0","ndarray[float64](6, 11)","[[0.83,3.16,1.4 ,...,1.02,0.17,0.68], [1. ,1.25,1.07,...,1.76,2.15,0.85], [0.83,0.89,0.88,...,0.99,1.04,0.48], [1.07,0.75,0.98,...,0.91,0.93,0.93], [1.12,0.62,0.95,...,0.88,0.72,0.81], [1.19,0.67,0.98,...,1.54,0.48,1.14]]"


In [587]:
y_pred3=modelo3.predict(X_test)

In [588]:
accuracy3 = accuracy_score(y_test, y_pred3)
matrix3 = confusion_matrix(y_test, y_pred3)
print(accuracy3)
print(classification_report(y_test, y_pred3))
print(matrix3)

0.345
              precision    recall  f1-score   support

           3       0.00      0.00      0.00         2
           4       0.04      0.08      0.05        13
           5       0.69      0.45      0.54       170
           6       0.51      0.21      0.30       160
           7       0.31      0.48      0.38        50
           8       0.06      0.60      0.10         5

    accuracy                           0.34       400
   macro avg       0.27      0.30      0.23       400
weighted avg       0.54      0.34      0.40       400

[[ 0  1  1  0  0  0]
 [ 9  1  2  1  0  0]
 [36 10 76 28 15  5]
 [20 13 31 34 37 25]
 [ 1  0  0  4 24 21]
 [ 0  0  0  0  2  3]]


## 6. Transformación a problema binario

Se agrupan las calidades en dos categorías:

- `0`: calidad menor que 6.
- `1`: calidad mayor o igual que 6.

A partir de aquí, las comparaciones principales se realizan sobre este problema binario.

In [589]:
y2=y
y2[y2<6]=0
y2[y2>=6]=1
modelo4=NaiveBayes()
X_train2, X_test2, y_train2, y_test2 = train_test_split(X, y2, test_size=0.25, random_state=42, stratify=y2)
modelo4.fit(X_train2, y_train2)

In [590]:
y_pred4=modelo4.predict(X_test2)

In [591]:
accuracy4 = accuracy_score(y_test2, y_pred4)
matrix4 = confusion_matrix(y_test2, y_pred4)
print(accuracy4)
print(classification_report(y_test2, y_pred4))
print(matrix4)

0.7225
              precision    recall  f1-score   support

           0       0.68      0.77      0.72       186
           1       0.77      0.68      0.72       214

    accuracy                           0.72       400
   macro avg       0.73      0.73      0.72       400
weighted avg       0.73      0.72      0.72       400

[[143  43]
 [ 68 146]]


## 7. Naive Bayes binario + Random Oversampling

Se aplica Random Oversampling al problema binario y se evalúa nuevamente Naive Bayes.

In [592]:
X_train_ros2, y_train_ros2 = ros.fit_resample(X_train2,y_train2)

In [593]:
modelo5=NaiveBayes()
modelo5.fit(X_train_ros2, y_train_ros2)

In [594]:
y_pred5=modelo5.predict(X_test2)

In [595]:
accuracy5 = accuracy_score(y_test2, y_pred5)
matrix5 = confusion_matrix(y_test2, y_pred5)
print(accuracy5)
print(classification_report(y_test2, y_pred5))
print(matrix5)

0.725
              precision    recall  f1-score   support

           0       0.68      0.77      0.72       186
           1       0.78      0.68      0.73       214

    accuracy                           0.72       400
   macro avg       0.73      0.73      0.72       400
weighted avg       0.73      0.72      0.73       400

[[144  42]
 [ 68 146]]


## 8. Naive Bayes binario + SMOTE

Se generan observaciones sintéticas de la clase minoritaria mediante SMOTE y se evalúa Naive Bayes.

In [596]:
smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(X_train2,y_train2)

In [597]:
modelo6=NaiveBayes()
modelo6.fit(X_train_smote, y_train_smote)
y_pred6=modelo6.predict(X_test2)

In [598]:
accuracy6 = accuracy_score(y_test2, y_pred6)
matrix6 = confusion_matrix(y_test2, y_pred6)
print(accuracy6)
print(classification_report(y_test2, y_pred6))
print(matrix6)

0.715
              precision    recall  f1-score   support

           0       0.66      0.78      0.72       186
           1       0.78      0.65      0.71       214

    accuracy                           0.71       400
   macro avg       0.72      0.72      0.71       400
weighted avg       0.72      0.71      0.71       400

[[146  40]
 [ 74 140]]


## 9. XGBoost sin SMOTE

Se entrena XGBoost sobre los datos de entrenamiento del problema binario y se evalúa sobre el conjunto de prueba.

In [599]:
xgb = XGBRegressor(objective='binary:logistic', random_state=42)
xgb.fit(X_train2, y_train2)
y_prob = xgb.predict(X_test2)
y_pred7 = np.where(y_prob >= 0.5, 1, 0)


In [600]:
accuracy7 = accuracy_score(y_test2, y_pred7)
matrix7 = confusion_matrix(y_test2, y_pred7)
print(accuracy7)
print(classification_report(y_test2, y_pred7))
print(matrix7)

0.8
              precision    recall  f1-score   support

           0       0.78      0.78      0.78       186
           1       0.81      0.81      0.81       214

    accuracy                           0.80       400
   macro avg       0.80      0.80      0.80       400
weighted avg       0.80      0.80      0.80       400

[[146  40]
 [ 40 174]]


## 10. XGBoost + SMOTE

Se reutiliza el modelo XGBoost y se entrena con el conjunto de entrenamiento balanceado mediante SMOTE. La evaluación se realiza sobre el conjunto de prueba original, sin SMOTE.

In [601]:
xgb.fit(X_train_smote, y_train_smote)

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegress

In [602]:
y_prob2 = xgb.predict(X_test2)
y_pred8 = np.where(y_prob2 >= 0.5, 1, 0)


In [603]:
accuracy8 = accuracy_score(y_test2, y_pred8)
matrix8 = confusion_matrix(y_test2, y_pred8)
print(accuracy8)
print(classification_report(y_test2, y_pred8))
print(matrix8)

0.805
              precision    recall  f1-score   support

           0       0.78      0.81      0.79       186
           1       0.83      0.80      0.82       214

    accuracy                           0.81       400
   macro avg       0.80      0.81      0.80       400
weighted avg       0.81      0.81      0.81       400

[[150  36]
 [ 42 172]]


## 11. Validación cruzada estratificada: XGBoost + SMOTE

Esta es la evaluación más importante del notebook. Se utiliza un `Pipeline` que contiene SMOTE y XGBoost, de modo que SMOTE se ejecuta dentro de cada fold únicamente sobre los datos de entrenamiento de ese fold.

Se utilizan **5 folds**, `shuffle=True` y `random_state=42`. Las métricas son Accuracy, F1 macro y Balanced Accuracy.

In [604]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

xgb = XGBClassifier(
    random_state=42
)

modelo_smote = Pipeline([
    ("smote", SMOTE(random_state=42)),
    ("xgb", xgb)
])

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_validate(
    modelo_smote,
    X,
    y2,
    cv=cv,
    scoring=[
        "accuracy",
        "f1_macro",
        "balanced_accuracy"
    ]
)

## 12. Resultados de la validación cruzada

La tabla muestra el resultado obtenido en cada uno de los 5 folds. El desempeño final puede resumirse mediante el promedio y la desviación estándar de cada métrica.

In [605]:
scores=pd.DataFrame(scores)
scores

,fit_time,score_time,test_accuracy,test_f1_macro,test_balanced_accuracy
0,0.073683,0.006397,0.775000,0.773931,0.773931
1,0.067452,0.005867,0.837500,0.837093,0.838023
2,0.083688,0.005929,0.803125,0.802800,0.804133
3,0.077953,0.006570,0.821875,0.819960,0.818655
4,0.065660,0.005720,0.752351,0.752341,0.755828


## Conclusión

El notebook queda organizado en tres bloques principales: **Naive Bayes**, **balanceo mediante oversampling/SMOTE** y **XGBoost**, terminando con una **validación cruzada estratificada de 5 folds** para XGBoost + SMOTE. Las celdas de código originales se conservaron; únicamente se añadieron encabezados y explicaciones para identificar qué corresponde a cada etapa.